In [ ]:
import pandas as pd
import os
import re
import glob
from pathlib import Path
import sys

import pickle
from pathlib import Path



In [ ]:
#working directories

data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-12-02_create_input_dataframes_for_matchit" 

results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-02_create_input_dataframes_for_matchit" 
results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-18_grouping_cohorts_by_vax_ancestry_season"


scratch = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-02_create_input_dataframes_for_matchit"



In [ ]:
#### load pkl data frames

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
#### wrangle data frames

In [ ]:

def wrangle_control_cohort_data(df):
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 1. Merge Race/Ethnicity ---
    
    # Define the helper function for merging
    def merge_race_ethnicity_data(row):
        """Helper function to apply row-wise."""
        if row["ethnicity"] == "Hispanic or Latino":
            return row["ethnicity"]
        else:
            return row["race"]

    # Apply the function to create the 'updated_race' column
    wrangled_df["updated_race"] = wrangled_df.apply(merge_race_ethnicity_data, axis=1)

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]


    return wrangled_df

In [ ]:

def wrangle_ns_cohorts_data(df):
    
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()


    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]

    # --- 3. Calculate Age ---
    
    # Ensure date columns are datetime objects, handling potential errors
    # Using .loc to avoid a potential SettingWithCopyWarning
    wrangled_df.loc[:, 'first_diagnosis_date'] = pd.to_datetime(wrangled_df['first_diagnosis_date'], errors="coerce")
    wrangled_df.loc[:, 'date_of_birth'] = pd.to_datetime(wrangled_df['date_of_birth'], errors="coerce")

    # Calculate age in years
    # This will result in NaN if either date was invalid (became NaT)
    wrangled_df['age'] = (wrangled_df['first_diagnosis_date'] - wrangled_df['date_of_birth']).dt.days / 365.25

    return wrangled_df

In [ ]:

def wrangle_vax_cohort_data(df):
    """
    Wrangles the cohort DataFrame by merging race/ethnicity,
    filtering specific groups, and calculating age.
    
    Args:
        df (pd.DataFrame): The input DataFrame. Must contain 
                         'ethnicity', 'race', 'first_diag_date_x', 
                         and 'date_of_birth' columns.

    Returns:
        pd.DataFrame: The wrangled DataFrame.
    """
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]

    # --- 3. Calculate Age ---
    
    # Ensure date columns are datetime objects, handling potential errors
    # Using .loc to avoid a potential SettingWithCopyWarning
    wrangled_df.loc[:, 'first_dx'] = pd.to_datetime(wrangled_df['first_dx'], errors="coerce")
    wrangled_df.loc[:, 'date_of_birth'] = pd.to_datetime(wrangled_df['date_of_birth'], errors="coerce")

    # Calculate age in years
    # This will result in NaN if either date was invalid (became NaT)
    wrangled_df['age'] = (wrangled_df['first_dx'] - wrangled_df['date_of_birth']).dt.days / 365.25

    return wrangled_df


In [ ]:
def get_wrangled_ns_cohorts_data(ns_dict_df):
    
    
    upd_ns_cases = {}

    for key, table in ns_dict_df.items():

        tmp = wrangle_ns_cohorts_data(table)

        upd_ns_cases[key] = tmp
        
    return upd_ns_cases

In [ ]:
def get_wrangled_vax_cohort_data(ns_vax_dict_df):
    
    upd_ns_vax_cases = {}

    for key, table in ns_vax_dict_df.items():
    
        tmp = wrangle_vax_cohort_data(table)

        upd_ns_vax_cases[key] = tmp
        
    return upd_ns_vax_cases

In [ ]:
#### get control age

In [ ]:

def calculate_control_age_at_index_date(
    cases_df,
    controls_df,
    output_csv_path,
    id_cases_col="person_id",
    case_dx_date_col="first_diagnosis_date",
    ctrl_dob_col="date_of_birth"
):
    
    """
    Calculates control ages based on the median case diagnosis date (index date)
    and saves the resulting control DataFrame to a CSV.
    
    Args:
        cases_df (pd.DataFrame): DataFrame of the case cohort.
        controls_df (pd.DataFrame): DataFrame of the control cohort.
        output_csv_path (str): File path to save the new control CSV.
        id_cases_col (str): Column name for case IDs in cases_df.
        case_dx_date_col (str): Column name for diagnosis dates in cases_df.
        ctrl_dob_col (str): Column name for date of birth in controls_df.
        
    Returns:
        pd.DataFrame: The modified control DataFrame with the new 'age' column.
    """
    
    
    
    
    # Make copies to avoid modifying the original DataFrames
    cases = cases_df.copy()
    ctrls = controls_df.copy()

    # 1) First diagnosis per case, then the cohort-wide median of those dates
    cases[case_dx_date_col] = pd.to_datetime(cases[case_dx_date_col], errors="coerce")
    first_dx = (cases
                .dropna(subset=[case_dx_date_col])
                .sort_values(case_dx_date_col)
                .groupby(id_cases_col, as_index=False)[case_dx_date_col].first())

    index_date = first_dx[case_dx_date_col].median()  # single anchor date
    print(f"Index date (median first diagnosis): {index_date.date()}")

    # 2) Controls: age at the index date (from DOB)
    ctrls[ctrl_dob_col] = pd.to_datetime(ctrls[ctrl_dob_col], errors="coerce")
    ctrls["age"] = (index_date - ctrls[ctrl_dob_col]).dt.days / 365.25

    # 3) Save to CSV (matches the original snippet's lack of index=False)
    ctrls.to_csv(output_csv_path)
    
    print(f"Control cohort with ages saved to: {output_csv_path}")
    
    # 4) Return the DataFrame for further use
    return ctrls

In [ ]:
def slugify(text):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    return text

In [ ]:
def slugify2(text, text2):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    
    text2 = str(text2).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text2 = re.sub(r'[^a-z0-9_]+', '_', text2) 
    text2 = text2.strip('_') # Clean up leading/trailing underscores
    
    return text, text2

In [ ]:
def slugify3(text, text2, text3, text4):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    
    text2 = str(text2).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text2 = re.sub(r'[^a-z0-9_]+', '_', text2) 
    text2 = text2.strip('_') # Clean up leading/trailing underscores
    
    
    text3 = str(text3).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text3 = re.sub(r'[^a-z0-9_]+', '_', text3) 
    text3 = text3.strip('_') # Clean up leading/trailing underscores
   

    text4 = str(text4).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text4 = re.sub(r'[^a-z0-9_]+', '_', text4) 
    text4 = text4.strip('_') # Clean up leading/trailing underscores



    return text, text2, text3, text4

In [ ]:
def slugify4(text, text2, text3):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    
    text2 = str(text2).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text2 = re.sub(r'[^a-z0-9_]+', '_', text2) 
    text2 = text2.strip('_') # Clean up leading/trailing underscores
    
    
    text3 = str(text3).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text3 = re.sub(r'[^a-z0-9_]+', '_', text3) 
    text3 = text3.strip('_') # Clean up leading/trailing underscores
   



    return text, text2, text3

In [ ]:
#NS cohort - getting controls age

def process_ns_cohorts(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify(concept_name)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=control1, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_diagnosis_date",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS bin cohort - getting controls age

def process_ns_bin_cohorts(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name, race), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify2(concept_name, race)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=control1, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_diagnosis_date",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS VAX cohorts - getting controls age


def process_ns_vax_cohorts(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify(concept_name)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=control1, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_dx",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS VAX  bin cohorts - getting controls age


def process_ns_vax_bin_cohorts(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name, race,), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify2(concept_name, race,)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=control1, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_dx",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS VAX flu  cohorts - getting controls age

def process_ns_vax_bin_cohorts2(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name, year, race, vax), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify3(concept_name, year, race, vax)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=control1, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_dx",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS VAX flu  cohorts - getting controls age

def process_ns_vax_bin_cohorts3(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name, year, race), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify4(concept_name, year, race)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=control1, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_dx",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#### EXPORT raw cohort DF AS CSVs

In [ ]:

def save_dataframes_to_csv(df_dict, 
                           output_folder="cohort_csv_output", 
                           suffix="_case_cohort"):
    """
    Saves each DataFrame in a dictionary to its own CSV file.

    The dictionary key is assumed to be a tuple: (concept_id, concept_name).
    The filename will be generated from the 'concept_name'.

    Args:
        df_dict (dict): The dictionary of { (id, name): df } to save.
                        (e.g., your 'ns_df_upd' variable)
        output_folder (str): Name of the folder to save the CSVs.
        suffix (str): The suffix to add to the filename (e.g., "_controls").
    """
    
    # 1. Create the output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    print(f"Saving CSVs to folder: {output_folder}")
    
    count = 0
    # 2. Loop through the dictionary items
    for (concept_id, concept_name), df in df_dict.items():
        
        # 3. Create the automated filename
        safe_filename = f"{slugify(concept_name)}{suffix}.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)
        
        try:
            # 4. Save the DataFrame to CSV
            #    index=False is generally recommended for data exports
            df.to_csv(output_csv_path, index=False)
            print(f"Successfully saved: {safe_filename}")
            count += 1
        except Exception as e:
            print(f"ERROR saving {safe_filename}: {e}", file=sys.stderr)
    
    print(f"\n--- Done. Saved {count} CSV files. ---")



In [ ]:

def save_bin_dataframes_to_csv(df_dict, 
                           output_folder="cohort_csv_output", 
                           suffix="_case_cohort"):
    """
    Saves each DataFrame in a dictionary to its own CSV file.

    The dictionary key is assumed to be a tuple: (concept_id, concept_name).
    The filename will be generated from the 'concept_name'.

    Args:
        df_dict (dict): The dictionary of { (id, name): df } to save.
                        (e.g., your 'ns_df_upd' variable)
        output_folder (str): Name of the folder to save the CSVs.
        suffix (str): The suffix to add to the filename (e.g., "_controls").
    """
    
    # 1. Create the output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    print(f"Saving CSVs to folder: {output_folder}")
    
    count = 0
    # 2. Loop through the dictionary items
    for (concept_id, concept_name, vax), df in df_dict.items():
        
        # 3. Create the automated filename
        safe_filename = f"{slugify2(concept_name, vax)}{suffix}.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)
        
        try:
            # 4. Save the DataFrame to CSV
            #    index=False is generally recommended for data exports
            df.to_csv(output_csv_path, index=False)
            print(f"Successfully saved: {safe_filename}")
            count += 1
        except Exception as e:
            print(f"ERROR saving {safe_filename}: {e}", file=sys.stderr)
    
    print(f"\n--- Done. Saved {count} CSV files. ---")



In [ ]:

def save_bin_dataframes_to_csv2(df_dict, 
                           output_folder="cohort_csv_output", 
                           suffix="_case_cohort"):
    """
    Saves each DataFrame in a dictionary to its own CSV file.

    The dictionary key is assumed to be a tuple: (concept_id, concept_name).
    The filename will be generated from the 'concept_name'.

    Args:
        df_dict (dict): The dictionary of { (id, name): df } to save.
                        (e.g., your 'ns_df_upd' variable)
        output_folder (str): Name of the folder to save the CSVs.
        suffix (str): The suffix to add to the filename (e.g., "_controls").
    """
    
    # 1. Create the output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    print(f"Saving CSVs to folder: {output_folder}")
    
    count = 0
    # 2. Loop through the dictionary items
    for (concept_id, concept_name, year, race, vax), df in df_dict.items():
        
        # 3. Create the automated filename
        safe_filename = f"{slugify3(concept_name,year, race, vax)}{suffix}.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)
        
        try:
            # 4. Save the DataFrame to CSV
            #    index=False is generally recommended for data exports
            df.to_csv(output_csv_path, index=False)
            print(f"Successfully saved: {safe_filename}")
            count += 1
        except Exception as e:
            print(f"ERROR saving {safe_filename}: {e}", file=sys.stderr)
    
    print(f"\n--- Done. Saved {count} CSV files. ---")



In [ ]:

def save_bin_dataframes_to_csv3(df_dict, 
                           output_folder="cohort_csv_output", 
                           suffix="_case_cohort"):
    """
    Saves each DataFrame in a dictionary to its own CSV file.

    The dictionary key is assumed to be a tuple: (concept_id, concept_name).
    The filename will be generated from the 'concept_name'.

    Args:
        df_dict (dict): The dictionary of { (id, name): df } to save.
                        (e.g., your 'ns_df_upd' variable)
        output_folder (str): Name of the folder to save the CSVs.
        suffix (str): The suffix to add to the filename (e.g., "_controls").
    """
    
    # 1. Create the output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    print(f"Saving CSVs to folder: {output_folder}")
    
    count = 0
    # 2. Loop through the dictionary items
    for (concept_id, concept_name, race, grp ), df in df_dict.items():
        
        # 3. Create the automated filename
        safe_filename = f"{slugify4(concept_name, race, grp)}{suffix}.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)
        
        try:
            # 4. Save the DataFrame to CSV
            #    index=False is generally recommended for data exports
            df.to_csv(output_csv_path, index=False)
            print(f"Successfully saved: {safe_filename}")
            count += 1
        except Exception as e:
            print(f"ERROR saving {safe_filename}: {e}", file=sys.stderr)
    
    print(f"\n--- Done. Saved {count} CSV files. ---")



In [ ]:
def save_dataframes_to_csv4(df_dict, 
                           output_folder="cohort_csv_output", 
                           suffix="_case_cohort"):
    """
    Saves each DataFrame in a dictionary to its own CSV file.

    The dictionary key is assumed to be a tuple: (concept_id, concept_name).
    The filename will be generated from the 'concept_name'.

    Args:
        df_dict (dict): The dictionary of { (id, name): df } to save.
                        (e.g., your 'ns_df_upd' variable)
        output_folder (str): Name of the folder to save the CSVs.
        suffix (str): The suffix to add to the filename (e.g., "_controls").
    """
    
    # 1. Create the output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    print(f"Saving CSVs to folder: {output_folder}")
    
    count = 0
    # 2. Loop through the dictionary items
    for (concept_name), df in df_dict.items():
        
        # 3. Create the automated filename
        safe_filename = f"{slugify(concept_name)}{suffix}.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)
        
        try:
            # 4. Save the DataFrame to CSV
            #    index=False is generally recommended for data exports
            df.to_csv(output_csv_path, index=False)
            print(f"Successfully saved: {safe_filename}")
            count += 1
        except Exception as e:
            print(f"ERROR saving {safe_filename}: {e}", file=sys.stderr)
    
    print(f"\n--- Done. Saved {count} CSV files. ---")

In [ ]:
def prepare_case_control_data_py(cases_df: pd.DataFrame,
                                controls_df: pd.DataFrame,
                                col_sex: str = "sex_at_birth",
                                col_ancestry: str = "updated_race",
                                col_age: str = "age") -> pd.DataFrame:
    """
    Prepares case and control data for analysis by combining, standardizing 
    variable types, and performing initial sanity checks.

    Args:
        cases_df: DataFrame containing the 'case' subjects.
        controls_df: DataFrame containing the 'control' subjects.
        col_sex: Name of the column representing sex.
        col_ancestry: Name of the column representing ancestry/race.
        col_age: Name of the column representing age.

    Returns:
        A single combined pandas DataFrame with 'case', 'sex', 'ancestry', 
        and 'age' columns standardized.
    """
    
    # 1) Harmonize column names and add case/control indicator
    # (Similar to dplyr::mutate and adding a fixed column)
    cases_df = cases_df.copy()
    controls_df = controls_df.copy()
    
    # Use 1 for Cases (int/integer)
    cases_df['case'] = 1 
    # Use 0 for Controls (int/integer)
    controls_df['case'] = 0

    # 2) Bind, clean types
    # (Similar to dplyr::bind_rows, using pd.concat)
    df = pd.concat([cases_df, controls_df], ignore_index=True)
    
    # 3) Mutate new columns using the string variables (column mapping)
    # This step renames and casts the specified columns to the desired types.
    df['sex'] = df[col_sex].astype('category') # Factors are equivalent to categories
    df['ancestry'] = df[col_ancestry].astype('category')
    df['age'] = pd.to_numeric(df[col_age], errors='coerce') # Similar to as.numeric

    # Optional: Drop rows where the critical new variables are missing 
    # (The R code didn't explicitly drop missing, but this is often good practice)
    # df = df.dropna(subset=['sex', 'ancestry', 'age', 'case']) 

    # 4) Quick sanity checks (prints to the console)
    print("--- Sanity Checks ---")
    
    # Frequency table for Cases vs. Controls (table(df$case))
    print("\nCases vs. Controls:")
    print(df['case'].value_counts())
    
    # Cross-tabulation for Sex vs. Case (table(df$sex, df$case))
    print("\nSex vs. Case:")
    print(pd.crosstab(df['sex'], df['case']))
    
    # Cross-tabulation for Ancestry vs. Case (table(df$ancestry, df$case))
    print("\nAncestry vs. Case:")
    print(pd.crosstab(df['ancestry'], df['case']))
    
    # Summary for Age (summary(df$age))
    print("\nAge Summary:")
    print(df['age'].describe())
    print("---------------------")

    # 5) Return the combined data frame
    return df

In [ ]:
# Assuming the prepare_case_control_data_py function is defined as above:
# from your_module import prepare_case_control_data_py 

def concat_matched_case_and_controls(case_folder, control_folder):
    """
    Loops through case cohort CSV files, finds the matching control cohort 
    file, and processes them using the preparation function.

    Args:
        case_folder (str): Path to the folder containing case CSV files.
        control_folder (str): Path to the folder containing control CSV files.

    Returns:
        dict: A dictionary where keys are cohort names (e.g., 'acute_bronchiolitis_due_to_respiratory_syncytial_virus_white')
              and values are the combined, processed pandas DataFrames.
    """
    processed_data = {}
    
    # 1. Use glob to find all case files
    case_files = glob.glob(os.path.join(case_folder, "*_case_cohort.csv"))
    
    if not case_files:
        print(f"No case files found in: {case_folder}")
        return processed_data

    print(f"Found {len(case_files)} case cohort files to process.")

    # 2. Loop through each case file
    for case_path in case_files:
        # Extract the base name (the common part of the file name)
        # e.g., 'acute_bronchiolitis_due_to_respiratory_syncytial_virus', 'white'
        base_name = os.path.basename(case_path).replace('_case_cohort.csv', '')
        
        # 3. Construct the corresponding control file name
        # The structure is: base_name + '_controls.csv'
        control_file_name = base_name + '_controls.csv'
        control_path = os.path.join(control_folder, control_file_name)
        
        # 4. Check if the control file exists
        if not os.path.exists(control_path):
            print(f"⚠️ Skipping: Control file not found for {base_name} at {control_path}")
            continue

        # 5. Read the files and process
        print(f"\n✅ Processing Cohort: {base_name}...")
        
        try:
            # Read DataFrames
            cases_df = pd.read_csv(case_path)
            controls_df = pd.read_csv(control_path)
            
            # Use the preparation function
            # NOTE: Assuming prepare_case_control_data_py is defined or imported
            combined_df = prepare_case_control_data_py(
                cases_df, 
                controls_df, 
                col_sex="sex_at_birth", # Use your actual column names
                col_ancestry="updated_race",
                col_age="age"
            )
            
            # Clean up the key name for the dictionary
            # Removes the tuple parentheses and spaces to make a cleaner key
            dict_key = base_name.strip("()").replace("'", "").replace(", ", "_")
            processed_data[dict_key] = combined_df
            
            print(f"   -> Successfully combined and processed {len(combined_df)} records.")

        except Exception as e:
            print(f"❌ Error processing {base_name}: {e}")
            
    return processed_data

In [ ]:
## function calls

##import all cohort PKLs


control = get_data_pkl(data,'control_cohort_demo_df.pkl')
ns_df = get_data_pkl(results1, 'ns_cohort_dict_df.pkl')
ns_vax_df =get_data_pkl(results1, 'ns_vax_cohort_dict_df.pkl')


ns_B = get_data_pkl(results1, 'b1_b2_dict_df.pkl')

ns_C = get_data_pkl(results1, 'c1_c2_dict_df.pkl')
ns_D = get_data_pkl(results1, 'd1_d2_dict_df.pkl')

ns_raw_vax_df = get_data_pkl(results1, 'ns_raw_vax_cohort_dict_df.pkl')

covid = get_data_pkl(results1, 'covid_bin_cohorts.pkl')
flu = get_data_pkl(results1, 'flu_bin_cohorts.pkl')


##Wrangle data

control1 = wrangle_control_cohort_data(control)

ns_df1 = get_wrangled_ns_cohorts_data(ns_df)
ns_B1 =  get_wrangled_ns_cohorts_data(ns_B)

ns_vax_df1 = get_wrangled_vax_cohort_data(ns_vax_df)
ns_raw_vax_df1 = get_wrangled_vax_cohort_data(ns_raw_vax_df)
ns_C1 = get_wrangled_vax_cohort_data(ns_C)
ns_D1 = get_wrangled_vax_cohort_data(ns_D)
covid1 = get_wrangled_vax_cohort_data(covid)
flu1 = get_wrangled_vax_cohort_data(flu)
                            
   

    
##NS cohorts


process_ns_cohorts(
        case_cohort_dict=ns_df1, 
        control_df_path=control1,
        output_folder= f"{scratch}/ns_controls"
    )
    


process_ns_bin_cohorts(
        case_cohort_dict=ns_B1, 
        control_df_path=control1,
        output_folder= f"{scratch}/b1_controls"
    )



##NS vax cohorts
process_ns_vax_cohorts(
        case_cohort_dict=ns_vax_df1, 
        control_df_path=control1,
        output_folder=f"{scratch}/ns_vax_controls"
    )




process_ns_vax_bin_cohorts(
        case_cohort_dict=ns_raw_vax_df1, 
        control_df_path=control1,
        output_folder=f"{scratch}/ns_raw_vax_controls"
    )






process_ns_vax_bin_cohorts2(
        case_cohort_dict=ns_C1, 
        control_df_path=control1,
        output_folder=f"{scratch}/c1_controls"
    )



process_ns_vax_bin_cohorts3(
        case_cohort_dict=ns_D1, 
        control_df_path=control1,
        output_folder=f"{scratch}/d1_controls"
    )



process_ns_vax_bin_cohorts2(
        case_cohort_dict=covid1, 
        control_df_path=control1,
        output_folder=f"{scratch}/covid_controls"
    )



process_ns_vax_bin_cohorts2(
        case_cohort_dict=flu1, 
        control_df_path=control1,
        output_folder=f"{scratch}/flu_controls"
    )







##export Raw cohorts





#NS cases
save_dataframes_to_csv(
         df_dict=ns_df1, 
         output_folder=f"{scratch}/ns_cases",
         suffix="_case_cohort"
     )



save_dataframes_to_csv(
         df_dict=ns_vax_df1, 
         output_folder=f"{scratch}/ns_vax_cases",
         suffix="_case_cohort"
     )





save_bin_dataframes_to_csv(
         df_dict=ns_raw_vax_df1, 
         output_folder=f"{scratch}/ns_raw_vax_cases",
         suffix="_case_cohort",
    
    )




save_bin_dataframes_to_csv(
         df_dict=ns_B1, 
         output_folder=f"{scratch}/b1_cases",
         suffix="_case_cohort",
    
     )
 


save_bin_dataframes_to_csv2(
         df_dict=ns_C1, 
         output_folder=f"{scratch}/c1_cases",
         suffix="_case_cohort",
    
     )   



save_bin_dataframes_to_csv3(
         df_dict=ns_D1, 
         output_folder=f"{scratch}/d1_cases",
         suffix="_case_cohort",
    
     )


save_bin_dataframes_to_csv2(
         df_dict=covid1, 
         output_folder=f"{scratch}/covid_cases",
         suffix="_case_cohort",
    
     )

save_bin_dataframes_to_csv2(
         df_dict=flu1, 
         output_folder=f"{scratch}/flu_cases",
         suffix="_case_cohort",
    
     )
     
ns = concat_matched_case_and_controls(f"{scratch}/ns_cases", f"{scratch}/ns_controls")
b1 = concat_matched_case_and_controls(f"{scratch}/b1_cases", f'{scratch}/b1_controls')
ns_vax = concat_matched_case_and_controls(f"{scratch}/ns_vax_cases", f"{scratch}/ns_vax_controls")
ns_vax_raw = concat_matched_case_and_controls(f"{scratch}/ns_raw_vax_cases", f"{scratch}/ns_raw_vax_controls")

c1 = concat_matched_case_and_controls(f"{scratch}/c1_cases", f"{scratch}/c1_controls")
d1 = concat_matched_case_and_controls(f"{scratch}/d1_cases", f"{scratch}/d1_controls")
cov = concat_matched_case_and_controls(f"{scratch}/covid_cases", f"{scratch}/covid_controls")
influenza = concat_matched_case_and_controls(f"{scratch}/flu_cases", f"{scratch}/flu_controls")


save_dataframes_to_csv4(
         df_dict=ns, 
         output_folder=f"{results}/ns_matchit",
         suffix="_matchit_cohort"
     )



save_dataframes_to_csv4(
         df_dict=ns_vax, 
         output_folder=f"{results}/ns_vax_matchit",
         suffix="_matchit_cohort"
     )





save_dataframes_to_csv4(
         df_dict=ns_vax_raw, 
         output_folder=f"{results}/ns_raw_vax_matchit",
         suffix="_matchit_cohort",
    
    )




save_dataframes_to_csv4(
         df_dict=b1, 
         output_folder=f"{results}/b1_matchit",
         suffix="_matchit_cohort",
    
     )
 


save_dataframes_to_csv4(
         df_dict=c1, 
         output_folder=f"{results}/c1_matchit",
         suffix="_matchit_cohort",
    
     )   



save_dataframes_to_csv4(
         df_dict=d1, 
         output_folder=f"{results}/d1_matchit",
         suffix="_matchit_cohort",
    
     )


save_dataframes_to_csv4(
         df_dict=cov, 
         output_folder=f"{results}/covid_matchit",
         suffix="_matchit_cohort",
    
     )

save_dataframes_to_csv4(
         df_dict=influenza, 
         output_folder=f"{results}/flu_matchit",
         suffix="_matchit_cohort",
    
     )
     




In [ ]:
#function call

ns = concat_matched_case_and_controls(f"{scratch}/ns_cases", f"{scratch}/ns_controls")
b1 = concat_matched_case_and_controls(f"{scratch}/b1_cases", f'{scratch}/b1_controls')
ns_vax = concat_matched_case_and_controls(f"{scratch}/ns_vax_cases", f"{scratch}/ns_vax_controls")
ns_vax_raw = concat_matched_case_and_controls(f"{scratch}/ns_raw_vax_cases", f"{scratch}/ns_raw_vax_controls")

c1 = concat_matched_case_and_controls(f"{scratch}/c1_cases", f"{scratch}/c1_controls")
d1 = concat_matched_case_and_controls(f"{scratch}/d1_cases", f"{scratch}/d1_controls")
cov = concat_matched_case_and_controls(f"{scratch}/covid_cases", f"{scratch}/covid_controls")
influenza = concat_matched_case_and_controls(f"{scratch}/flu_cases", f"{scratch}/flu_controls")



In [ ]:
#function call



save_dataframes_to_csv4(
         df_dict=ns_vax_raw, 
         output_folder=f"{results}/ns_raw_vax_matchit",
         suffix="_matchit_cohort",
    
    )




save_dataframes_to_csv4(
         df_dict=b1, 
         output_folder=f"{results}/b1_matchit",
         suffix="_matchit_cohort",
    
     )
 


save_dataframes_to_csv4(
         df_dict=c1, 
         output_folder=f"{results}/c1_matchit",
         suffix="_matchit_cohort",
    
     )   



save_dataframes_to_csv4(
         df_dict=d1, 
         output_folder=f"{results}/d1_matchit",
         suffix="_matchit_cohort",
    
     )


save_dataframes_to_csv4(
         df_dict=cov, 
         output_folder=f"{results}/covid_matchit",
         suffix="_matchit_cohort",
    
     )

save_dataframes_to_csv4(
         df_dict=influenza, 
         output_folder=f"{results}/flu_matchit",
         suffix="_matchit_cohort",
    
     )
     

